# AK-SYS basic example

Three-component AND-system example from Fauriat and Gayton (2014). AK-SYS fits one Gaussian-process surrogate per component and evaluates only the component that controls the system U-function at each added point.


In [ ]:
import numpy as np
from ak_based_sampler import AK_SYS


def g1(u):
    return 8 * u[..., 1]**2 - 8 * u[..., 0]**2 + (u[..., 0]**2 + u[..., 1]**2)**2


def g2(u):
    return 2 * u[..., 0]**2 - 2 * u[..., 1]**2 - (u[..., 0]**2 + u[..., 1]**2)**2


def g3(u):
    return 8 * u[..., 1]**2 - 8 * u[..., 0]**2 - (u[..., 0]**2 + u[..., 1]**2)**2


In [ ]:
sampler = AK_SYS(dim=2, LSFs=[g1, g2, g3], gate="AND")

sampler.set_parameters(
    n_initial_population=10_000,
    n_initial_DoEs=12,
    u_stop=2.0,
    cov_stop=0.05,
    max_iterations_u=100,
    max_iterations_cov=10,
)

pf = sampler(seed=1, verbose=True)
print(f"Estimated system failure probability: {pf:.6g}")
print(dict(sampler.result))


In [ ]:
import matplotlib.pyplot as plt

population_u, component_g, component_sigma, system_g = sampler.train_data["population"]
initial_does = sampler.train_data["initial_DoE"]
added_does = sampler.train_data["DoE"]

grid = np.linspace(-5, 5, 350)
u1, u2 = np.meshgrid(grid, grid)
grid_points = np.stack([u1, u2], axis=-1)

fig, ax = plt.subplots(figsize=(7, 6))
for func, style, label in zip([g1, g2, g3], ["-", "--", ":"], ["g1 = 0", "g2 = 0", "g3 = 0"]):
    ax.contour(u1, u2, func(grid_points), levels=[0.0], colors="black", linestyles=style, linewidths=1.4)
    ax.plot([], [], color="black", linestyle=style, label=label)

failure_mask = system_g <= 0.0
ax.scatter(population_u[~failure_mask, 0], population_u[~failure_mask, 1], s=3, c="#4c78a8", alpha=0.12, linewidths=0, label="Predicted safe population")
ax.scatter(population_u[failure_mask, 0], population_u[failure_mask, 1], s=3, c="#e45756", alpha=0.18, linewidths=0, label="Predicted system failure")

markers = ["o", "s", "^"]
for i, ((initial_u, _), (added_u, _)) in enumerate(zip(initial_does, added_does)):
    ax.scatter(initial_u[:, 0], initial_u[:, 1], s=40, marker=markers[i], c="#f2cf5b", edgecolors="black", linewidths=0.5, label=f"Initial DoE: g{i + 1}")
    if len(added_u):
        ax.scatter(added_u[:, 0], added_u[:, 1], s=42, marker=markers[i], c="#54a24b", edgecolors="black", linewidths=0.5, label=f"Added DoE: g{i + 1}")

ax.set_title("AK-SYS result: three-component AND system")
ax.set_xlabel("u1")
ax.set_ylabel("u2")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right", fontsize=8, ncols=2)
ax.grid(alpha=0.2)
plt.show()
